<a href="https://colab.research.google.com/github/muskanpandit0361-cmd/GenAI_Practicals/blob/main/Practical_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers

In [ ]:
import transformers

In [ ]:
text=input()
print (text)

I absolutely loved the trip because the weather was beautiful, the hotel room was clean and comfortable, the food was delicious, the staff were friendly and helpful, the tourist attractions were amazing, and we were able to visit all the wonderful places we had planned to see
I absolutely loved the trip because the weather was beautiful, the hotel room was clean and comfortable, the food was delicious, the staff were friendly and helpful, the tourist attractions were amazing, and we were able to visit all the wonderful places we had planned to see


In [ ]:
from transformers import pipeline
classifier= pipeline("text-classification")

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [ ]:
outputs=classifier(text)

In [ ]:
import pandas as pd
pd.DataFrame(outputs)

,label,score
0,POSITIVE,0.999891


In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Load a lightweight T5 model and its corresponding tokenizer
model_name = "t5-small"
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
tok = AutoTokenizer.from_pretrained(model_name)

# Prepare input text with the T5 summarization prefix
input_text = "summarize: " + text
inputs = tok(
    input_text,
    return_tensors="pt",
    max_length=512,
    truncation=True
)

# Generate summary
summary_ids = model.generate(
    inputs["input_ids"],
    max_length=56,
    min_length=10,
    length_penalty=2.0,
    num_beams=4,
    early_stopping=True
)

summary = tok.decode(
    summary_ids[0],
    skip_special_tokens=True,
    clean_up_tokenization_spaces=True
)

print([{"summary_text": summary}])

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

[{'summary_text': 'the weather was beautiful, the hotel room was clean and comfortable, the food was delicious, the staff were friendly and helpful, the tourist attractions were amazing. we were able to visit all the wonderful places we had planned to see.'}]


In [ ]:
import torch
from transformers import AutoModelForQuestionAnswering, AutoTokenizer

# Load a robust, lightweight QA model and tokenizer
qa_model_name = "distilbert-base-cased-distilled-squad"

qa_model = AutoModelForQuestionAnswering.from_pretrained(qa_model_name)
qa_tokenizer = AutoTokenizer.from_pretrained(qa_model_name)

# Define the question and context
question = "How was the room?"
context = text

# Tokenize inputs
inputs = qa_tokenizer(question, context, return_tensors="pt")

# Get model predictions
with torch.no_grad():
    outputs = qa_model(**inputs)

# Decode the answer
answer_start = torch.argmax(outputs.start_logits)
answer_end = torch.argmax(outputs.end_logits) + 1

answer = qa_tokenizer.convert_tokens_to_string(
    qa_tokenizer.convert_ids_to_tokens(
        inputs["input_ids"][0][answer_start:answer_end]
    )
)

print("Question:", question)
print("Answer:", answer)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Question: How was the room?
Answer: clean and comfortable
